# Effective dimension of the OBLIQ embeddings (mean pooled)

Participation ratio (PR) of single-vector BERT and ColBERT embeddings for every OBLIQ dataset, split (corpus, queries), encoder and vector set (tag) under `datasets/<name>/vec_{corpus,queries}/<encoder>/`. Missing combinations are skipped.

Each item (document or query) is one point: the mean of its stored token vectors (mean pooling). Several vector sets for one split (e.g. `obliq-writing` queries at `nv32` and `nv180`) each get a row. ColBERT query vectors include the `[MASK]` augmentation vectors, as stored, so they are part of each query's mean.

For centred points $x_i$ with covariance $C$, $\mathrm{PR} = (\operatorname{tr} C)^2 / \operatorname{tr}(C^2) = (\sum_k \lambda_k)^2 / \sum_k \lambda_k^2$. It ranges from 1 (one direction) to $d$ (isotropic). The computation lives in `effdim.py`: an exact one-pass accumulator with O(d²) memory (same value as `participation_ratio`), reading memory-mapped vectors in blocks, so it scales to billion-point sets. With `SAMPLE = m` it also estimates PR on `SAMPLE_SEEDS` random subsets of m points (mean ± sd); `EXACT = False` runs the sampling estimate alone (a set with at most m points always gets the full pass).

The covariance spectrum (eigenvalues of the centred d × d covariance, i.e. of $A^\top A$ for the centred data $A$) comes from the same sums at no extra I/O. It is exact after a full pass; otherwise it is the spectrum of the mean sample covariance, with a per-eigenvalue sd across samples. With m ≫ d (e.g. m = 10⁶, d ≤ 768) the sample spectrum is close to the full one.

For single-vector `.npy` datasets, see `effective_dim.ipynb`. Use the **aws** kernel.

In [ ]:
import pandas as pd

from effdim import estimate_pr, item_mean_blocks, open_multivector, spectrum_summary
from paths import DATASETS_DIR, RESULTS_DIR

DATASETS = ["obliq-math", "obliq-twitter", "obliq-writing"]
ENCODERS = ["colbert", "bert"]
SPLITS = ["corpus", "queries"]
CHUNK = 1 << 16      # token vectors read per block
EXACT = True         # full one-pass PR over every item
SAMPLE = None        # e.g. 10_000: also estimate PR from random subsets of this many items
SAMPLE_SEEDS = 5
SEED = 0

## Compute

In [ ]:
rows = []
for dataset in DATASETS:
    for encoder in ENCODERS:
        for split in SPLITS:
            folder = DATASETS_DIR / dataset / f"vec_{split}" / encoder
            tags = sorted(p.name.removesuffix(".meta.json") for p in folder.glob("*.meta.json"))
            if not tags:
                print(f"skip {dataset}/{split}/{encoder}: no vectors")
            for tag in tags:
                V, offsets, meta = open_multivector(folder / tag)
                n = len(offsets) - 1
                pr, lam, _ = estimate_pr(n, lambda items: item_mean_blocks(V, offsets, items, CHUNK),
                                         f"{dataset}/{split}/{tag}", EXACT, SAMPLE, SAMPLE_SEEDS, SEED)
                rows.append({"dataset": dataset, "encoder": encoder, "split": split, "tag": tag,
                             "dim": meta["dim"], "items": n, **pr, **spectrum_summary(lam)})

effdim = pd.DataFrame(rows)
for c in ("PR", "sample PR mean"):
    if c in effdim:
        effdim[c + " / d"] = effdim[c] / effdim["dim"]

## Results

`/ d` columns normalise by the embedding dimension (128 for ColBERT, 768 for BERT), so the two encoders can be compared. Sample columns appear only when `SAMPLE` is set and smaller than the number of items.

In [ ]:
effdim.style.format({c: "{:.1f}" for c in effdim.columns if "PR" in c and "/ d" not in c}
                    | {c: "{:.3f}" for c in effdim.columns if "/ d" in c}
                    | {"items": "{:,}", "sample size": "{:,.0f}"}, na_rep="–").hide(axis="index")

In [ ]:
# Side by side: one row per dataset/split/vector set, one column per encoder.
pr_cols = [c for c in ("PR", "PR / d", "sample PR mean", "sample PR mean / d") if c in effdim]
(effdim.assign(max_len=effdim.tag.str.extract(r"_nv(\d+)$", expand=False))
       .pivot(index=["dataset", "split", "max_len"], columns="encoder", values=pr_cols))

In [ ]:
# effdim.to_csv(RESULTS_DIR / "obliq_effective_dimension.csv", index=False)